In [1]:
from appworld import AppWorld, load_task_ids
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import List, Literal, Optional, Tuple
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1",
    api_key="dummy",  # or your real key if you used --api-key
)

In [3]:
resp = client.chat.completions.create(
    model="microsoft/Phi-3-mini-4k-instruct",
    messages=[
        {"role": "user", "content": "Explain gravity in one sentence."}
    ],
    max_tokens=50,
    logprobs=True,          # <-- REQUIRED
)

In [4]:
from baseline.templates import Template
from baseline.config import Config

# models.py
from pydantic import BaseModel
from typing import List, Literal


class Message(BaseModel):
    """Single conversation message"""
    role: Literal["system", "user", "assistant"]
    content: str
    log_probs: Optional[List[Tuple[str, float]]] = None


class AgentState(BaseModel):
    conversation_history: List[Message] = Field(default_factory=list)
    log_probs: List[float] = Field(default_factory=list)
    iteration: int = 0
    done: bool = False
    max_iters: int = 50
    
    @property 
    def should_continue(self):
        return self.iteration < self.max_iters and not self.done
    
    def total_chars(self):
        return sum(len(msg.content) for msg in self.conversation_history)

In [5]:
import re

def message_parser_with_position(message: str) -> Tuple[Optional[str], Optional[int], Optional[int]]:
    """
    Extract code from markdown code blocks and return code + positions.
    Returns (code, start_pos, end_pos) where positions mark the full code block including ```.
    """
    pattern = r'```(?:python)?\n(.*?)```'
    match = re.search(pattern, message, re.DOTALL)
    
    if match:
        code = match.group(1).strip()
        # match.start() is the position of the opening ```
        # match.end() is the position after the closing ```
        return code, match.start(), match.end()
    
    # Fallback for non-markdown code
    if message.strip().startswith(('print(', 'apis.', 'import ', 'from ')):
        return message.strip(), 0, len(message)
    
    return None, None, None

def truncate_message_history(
    conversation_history: List[Message], 
    threshold: int
) -> List[Message]:
    """
    Truncate conversation history if it exceeds the character threshold.
    Keeps the system message and most recent messages.
    """
    total_chars = sum(len(msg.content) for msg in conversation_history)
    
    if total_chars <= threshold:
        return conversation_history
    
    # Always keep the first message (initial prompt with instructions)
    truncated = [conversation_history[0]]
    
    # Keep most recent messages until we're under threshold
    recent_messages = []
    current_chars = len(conversation_history[0].content)
    
    # Work backwards from most recent
    for msg in reversed(conversation_history[1:]):
        msg_chars = len(msg.content)
        if current_chars + msg_chars <= threshold:
            recent_messages.insert(0, msg)
            current_chars += msg_chars
        else:
            break
    
    truncated.extend(recent_messages)
    return truncated

In [6]:
from dataclasses import dataclass
import os
from dotenv import load_dotenv

load_dotenv()

@dataclass 
class Config:
    # Agent parameters
    max_iters: int = 50  # Match the paper's baseline
    
    # OpenAI parameters
    openai_api_key: str = os.getenv("OPENAI_API_KEY")
    service: str = "vLLM" # can set to OpenAI or TogetherAI
    togetherai_api_key: str = os.getenv("TOGETHER_AI")
    base_model: str = os.getenv("VLLM_MODEL") 
    max_tokens: int = 512
    temperature: float = 0.0
    
    # Context management
    truncation_threshold: int = 12000  # Characters, not tokens

    # AppWorld Root
    os.environ["APPWORLD_ROOT"] = os.getenv("APPWORLD_ROOT")
    
    @classmethod
    def for_model(cls, model_name: str):
        """Factory method for different model configs"""
        configs = {
            "gpt-4o": cls(base_model="gpt-4o-2024-05-13"),
            "gpt-4": cls(base_model="gpt-4-turbo-2024-04-09"),
            "o1": cls(
                base_model="o1-preview-2024-09-12",
                temperature=1.0,  # o1 requires temperature=1
                max_tokens=4000
            )
        }
        return configs.get(model_name, cls())

config = Config()

In [7]:
max_iters: int = config.max_iters
max_tokens: int = config.max_tokens
temperature: float = config.temperature
base_model: str = config.base_model
truncation_threshold: int = config.truncation_threshold
template = Template()
state = AgentState(max_iters=config.max_iters)
seed = None

In [8]:
task_ids = load_task_ids("train") # loads train ids, other options: dev, test_normal, test_challenge
task_id = task_ids[0]
world = AppWorld(task_id=task_id)

In [9]:
first_name = "John"
last_name = "Smith"
email = "john@yahoo.com"
phone_number = "1234567894"

init_template = template.format_prompt(
    first_name, 
    last_name, 
    email, 
    phone_number, 
    world.task.instruction
)

In [10]:
K=6
sets=["train"]

In [11]:
train_ids = [
		    tid
		    for dataset_name in sets
		    for tid in load_task_ids(dataset_name)
		]

In [12]:
task_id = train_ids[1]

In [13]:
import uuid
random_uuid = uuid.uuid4()

In [14]:
task_result = {
					"task_id": task_id,
					"completed": False,
					"iterations": 0,
					"error": None,
					"result": None,
					"conversation_length": 0,
					"token_log_probs": None,
					"unit_tests": None,
					"overall_success": None,
					"uuid": random_uuid
				}

In [15]:
from typing import Union
class ReactAgent:
    def __init__(self, config: Config, return_log_probs: bool = False, seed: int = None) -> None:
        self.max_iters: int = config.max_iters  # Fixed: use instance
        self.max_tokens: int = 512
        self.temperature: float = config.temperature
        self.base_model: str = config.base_model
        self.truncation_threshold: int = config.truncation_threshold
        self.template = Template()
        self.state = AgentState(max_iters=config.max_iters)
        self.seed = seed
        if config.service == "OpenAI":
            self.client = OpenAI(api_key=config.openai_api_key)
        elif config.service == "TogetherAI":
            os.environ["TOGETHER_API_KEY"] = config.togetherai_api_key
            self.client = OpenAI(
                api_key=config.togetherai_api_key,
                base_url="https://api.together.xyz/v1"
            )
        elif config.service == "vLLM":
            openai_api_key = "EMPTY"
            openai_api_base = "https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1"
            self.client = OpenAI(
                api_key=openai_api_key,
                base_url=openai_api_base,
            )
        self.eval_tracker: Dict = {}  

        
    def initialize(
        self, 
        first_name: str, 
        last_name: str, 
        email: str, 
        phone_number: str, 
        task_instructions: str
    ) -> None:
        init_template = self.template.format_prompt(
            first_name, 
            last_name, 
            email, 
            phone_number, 
            task_instructions
        )
        self.state.conversation_history.append(
            Message(role="user", content=init_template)
        )
    
    def call_llm(self, return_log_probs: bool = True) -> Tuple[str, Union[None, List[Tuple]]]: 
        messages = truncate_message_history(
            self.state.conversation_history, 
            self.truncation_threshold
        )

        extra_args = {}
        if self.seed:
            extra_args["seed"] = self.seed
        if return_log_probs:
            extra_args["logprobs"] = True
            extra_args["top_logprobs"] = 1  # only need the generated token

        response = self.client.chat.completions.create(
            model=self.base_model,
            messages=[msg.dict() for msg in messages],
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            **extra_args,
        )

        choice = response.choices[0]
        text = choice.message.content

        if not return_log_probs:
            return text, None

        token_logprobs = []
        if choice.logprobs is not None:
            for item in choice.logprobs.content:
                token_logprobs.append((item.token, item.logprob))

        return text, token_logprobs
    
    def step(self, world):  
        llm_output, token_logprobs = self.call_llm(return_log_probs=True)
        
        # Log full output for analysis
        self.eval_tracker[f"iter_{self.state.iteration}_full_output"] = llm_output
        
        # Extract code and find its position
        code, code_start, code_end = message_parser_with_position(llm_output)
        observation_string = "No code block found in response."
        
        if code:
            try:
                observation = world.execute(code)
                observation_string = str(observation)
            except Exception as e:
                observation_string = f"Error: {str(e)}"
            
            truncated_logprobs = []
            found_opening = False
            backtick_count = 0
            
            for i, (token, logprob) in enumerate(token_logprobs):
                truncated_logprobs.append((token, logprob))
                
                # Count backtick tokens
                if token == '```':
                    backtick_count += 1
                    if backtick_count == 1:
                        found_opening = True
                    elif backtick_count == 2 and found_opening:
                        # Found closing backticks - stop here
                        break
            
            self.state.conversation_history.append(
                Message(role="assistant", content=llm_output[:code_end].strip(), log_probs=truncated_logprobs)
            )
        else:
            # No code found - store full response
            self.state.conversation_history.append(
                Message(role="assistant", content=llm_output, log_probs=token_logprobs)
            )
        
        # Append real observation
        self.state.conversation_history.append(
            Message(role="user", content=f"Output:\n```\n{observation_string}\n```")
        )
        
        self.state.iteration += 1
        
        if world.task_completed():
            self.state.done = True
        
        return world
    
    def run(self, world):
        """
        Execute agent loop until completion or max iterations
        """
        while self.state.should_continue:
            world = self.step(world)
            
            if self.state.done:
                print(f"Task completed in {self.state.iteration} iterations")
                break
        
        if not self.state.done:
            print(f"Max iterations ({self.max_iters}) reached without completion")
        
        return world


In [16]:
agent = ReactAgent(config)

In [17]:
agent.initialize(
                first_name,
                last_name,
                email,
                phone_number,
                world.task.instruction
            )	

In [18]:
agent.run(world)

BadRequestError: Error code: 400 - {'error': {'message': "'max_tokens' or 'max_completion_tokens' is too large: 512. This model's maximum context length is 4096 tokens and your request has 3682 input tokens (512 > 4096 - 3682). None", 'type': 'BadRequestError', 'param': None, 'code': 400}}

C:\Users\Aaron McClendon\Documents\github\maml-agent\env311\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
print(agent.state.conversation_history[0].content)

In [ ]:
print(agent.state.conversation_history[1].content)

In [ ]:
agent.state.conversation_history[1].log_probs

In [ ]:
count = 0

for token, prob in agent.state.conversation_history[1].log_probs:
    count += len(token)

count

In [ ]:
extra_args = {}

extra_args["logprobs"] = True
extra_args["top_logprobs"] = 1  

response = agent.client.chat.completions.create(
    model=agent.base_model,
    messages=[{"role": "user", "content": str(agent.state.conversation_history[0].content)}],
    temperature=agent.temperature,
    max_tokens=agent.max_tokens,
    **extra_args
)

In [ ]:
choice = response.choices[0]
text = choice.message.content

token_logprobs = []
if choice.logprobs is not None:
    for item in choice.logprobs.content:
        token_logprobs.append((item.token, item.logprob))

In [ ]:
# Extract code and find its position
llm_output = text

code, code_start, code_end = message_parser_with_position(llm_output)
observation_string = "No code block found in response."

In [ ]:
code_end

In [ ]:
# Truncate to include everything up to and including the code block
truncated_response = llm_output[:code_end].strip()

In [ ]:
truncated_response

In [ ]:
# Build up tokens until exact match

truncated_logprobs = []
found_opening = False
backtick_count = 0

for i, (token, logprob) in enumerate(token_logprobs):
    truncated_logprobs.append((token, logprob))
    
    # Count backtick tokens
    if token == '```':
        backtick_count += 1
        if backtick_count == 1:
            found_opening = True
        elif backtick_count == 2 and found_opening:
            # Found closing backticks - stop here
            break


In [ ]:
truncated_logprobs

In [ ]:
for tup in truncated_logprobs:
    print(repr(tup[0]))

In [19]:
task_result["completed"] = world.task_completed()

In [20]:
task_result["iterations"] = agent.state.iteration

In [21]:
task_result["conversation_length"] = len(agent.state.conversation_history)

In [22]:
task_result["token_log_probs"] = [
						    msg.log_probs
						    for msg in agent.state.conversation_history
						    if msg.role == "assistant"
						]

In [23]:
task_result["assistant_messages"] = [
						    msg.content
						    for msg in agent.state.conversation_history
						    if msg.role == "assistant"
						]

In [31]:
world.safety_guard.disable()

results = world.evaluate()

────────────────────────────────────────────────── Overall Stats ──────────────────────────────────────────────────

Num Passed Tests : 1

Num Failed Tests : 1

Num Total  Tests : 2

───────────────────────────────────────────────────── Passes ──────────────────────────────────────────────────────

...

assert no model changes.

────────────────────────────────────────────────────── Fails ──────────────────────────────────────────────────────

>> Failed Requirement

assert answers match.

```python
with test(
    """
    assert answers match.
    """
):
    test.answer(predicted_answer, ground_truth_answer)
```
----------
AssertionError:  '<<not_given>>' == 'a love that never was'

In [34]:
type(results)

appworld.evaluator.TestTracker

In [37]:
results.pass_count

1

In [38]:
results.fail_count

1

In [41]:
outs = results.to_dict()

In [43]:
outs.keys()

dict_keys(['success', 'difficulty', 'num_tests', 'passes', 'failures'])

In [45]:
outs["num_tests"]

2

In [47]:
len(outs["passes"])

1

In [52]:
task_result["overall_success"] = len(evaluation['passes'])/evaluation['num_tests']

In [53]:
task_result["evaluation_details"] = evaluation

In [54]:
task_result

{'task_id': '82e2fac_2',
 'completed': False,
 'iterations': 15,
 'error': None,
 'result': None,
 'conversation_length': 31,
 'token_log_probs': [[('Okay', -0.7412142753601074),
   ('.', -0.3165111243724823),
   ('L', -0.9072583317756653),
   ('ets', -0.0003897384158335626),
   ('first', -0.3292463719844818),
   ('find', -0.06045417860150337),
   ('which', -0.16150923073291779),
   ('APIs', -0.044258084148168564),
   ('are', -0.0025912299752235413),
   ('available', -0.000567275274079293),
   ('to', -0.024503814056515694),
   ('use', -0.0010787388309836388),
   ('in', -0.10865531861782074),
   ('Sp', -0.043847426772117615),
   ('ot', -1.549708758830093e-05),
   ('ify', -1.1086402082582936e-05),
   ('.', -0.05449630692601204),
   ('\n', -0.03822099789977074),
   ('Code', -0.11245669424533844),
   (':', -0.0008985534077510238),
   ('\n', -0.0011038646334782243),
   ('```', -0.004141089040786028),
   ('python', -0.0005218812730163336),
   ('\n', -7.629103492945433e-05),
   ('print', -0.0

In [55]:
task_result["agent_state"] = agent.state

In [56]:
task_result["agent_state"]

AgentState(conversation_history=[Message(role='user', content='USER:\n    I am your supervisor and you are a super intelligent AI Assistant whose job is to achieve my day-to-day tasks completely autonomously.\n\n    To do this, you will need to interact with app/s (e.g., spotify, venmo etc) using their associated APIs on my behalf. \n    For this you will undertake a *multi-step conversation* using a python REPL environment. That is, you will write the \n    python code and the environment will execute it and show you the result, based on which, you will write python code \n    for the next step and so on, until you\'ve achieved the goal. This environment will let you interact with app/s using their associated APIs on my behalf.\n\n    Here are three key APIs that you need to know to get more information\n\n    # To get a list of apps that are available to you.\n    print(apis.api_docs.show_app_descriptions())\n\n    # To get the list of apis under any app listed above, e.g. spotify\n 

In [59]:
for message in range(len(task_result["agent_state"].conversation_history)):
    text = task_result["agent_state"].conversation_history[message]
    print("\n \n")
    print(f"{text.role}: {text.content}")


 

user: USER:
    I am your supervisor and you are a super intelligent AI Assistant whose job is to achieve my day-to-day tasks completely autonomously.

    To do this, you will need to interact with app/s (e.g., spotify, venmo etc) using their associated APIs on my behalf. 
    For this you will undertake a *multi-step conversation* using a python REPL environment. That is, you will write the 
    python code and the environment will execute it and show you the result, based on which, you will write python code 
    for the next step and so on, until you've achieved the goal. This environment will let you interact with app/s using their associated APIs on my behalf.

    Here are three key APIs that you need to know to get more information

    # To get a list of apps that are available to you.
    print(apis.api_docs.show_app_descriptions())

    # To get the list of apis under any app listed above, e.g. spotify
    print(apis.api_docs.show_api_descriptions(app_name='spotify'))

 

In [61]:
task_set = task_id

In [63]:
all_rollouts = []
all_rollouts.append(task_result)

In [64]:
all_rollouts[0]["task_id"]

'82e2fac_2'

In [66]:
rollout = all_rollouts[0]

In [67]:
rollout_reward = rollout["overall_success"]

In [68]:
rollout_id = rollout["uuid"]

In [69]:
baseline_reward = 1

In [70]:
LOO_advantage = rollout_reward - baseline_reward

In [71]:
rollout["advantage"] = LOO_advantage

In [72]:
updated_rollouts = []

In [73]:
updated_rollouts.append(rollout)

In [74]:
updated_rollouts

[{'task_id': '82e2fac_2',
  'completed': False,
  'iterations': 15,
  'error': None,
  'result': None,
  'conversation_length': 31,
  'token_log_probs': [[('Okay', -0.7412142753601074),
    ('.', -0.3165111243724823),
    ('L', -0.9072583317756653),
    ('ets', -0.0003897384158335626),
    ('first', -0.3292463719844818),
    ('find', -0.06045417860150337),
    ('which', -0.16150923073291779),
    ('APIs', -0.044258084148168564),
    ('are', -0.0025912299752235413),
    ('available', -0.000567275274079293),
    ('to', -0.024503814056515694),
    ('use', -0.0010787388309836388),
    ('in', -0.10865531861782074),
    ('Sp', -0.043847426772117615),
    ('ot', -1.549708758830093e-05),
    ('ify', -1.1086402082582936e-05),
    ('.', -0.05449630692601204),
    ('\n', -0.03822099789977074),
    ('Code', -0.11245669424533844),
    (':', -0.0008985534077510238),
    ('\n', -0.0011038646334782243),
    ('```', -0.004141089040786028),
    ('python', -0.0005218812730163336),
    ('\n', -7.629103492

In [76]:
rollout.keys()

dict_keys(['task_id', 'completed', 'iterations', 'error', 'result', 'conversation_length', 'token_log_probs', 'unit_tests', 'overall_success', 'uuid', 'assistant_messages', 'evaluation_details', 'agent_state', 'advantage'])

In [77]:
messages = rollout["agent_state"].conversation_history
assistant_messages = rollout["assistant_messages"]
token_log_probs_list = rollout["token_log_probs"]

In [81]:
messages[3].role

'assistant'

In [79]:
assistant_messages[1]

"I apologize for the confusion. It seems there was a misunderstanding. Let's proceed with the task at hand.\n\nTask: What is the title of the most-liked song in my Spotify playlists.\n\nASSISTANT:\nOkay. Lets first find which APIs are available to use in Spotify.\nCode:\n```python\nprint(apis.api_docs.show_api_descriptions(app_name='spotify'))\n```"

In [82]:
context = messages[:3]

In [83]:
context

[Message(role='user', content='USER:\n    I am your supervisor and you are a super intelligent AI Assistant whose job is to achieve my day-to-day tasks completely autonomously.\n\n    To do this, you will need to interact with app/s (e.g., spotify, venmo etc) using their associated APIs on my behalf. \n    For this you will undertake a *multi-step conversation* using a python REPL environment. That is, you will write the \n    python code and the environment will execute it and show you the result, based on which, you will write python code \n    for the next step and so on, until you\'ve achieved the goal. This environment will let you interact with app/s using their associated APIs on my behalf.\n\n    Here are three key APIs that you need to know to get more information\n\n    # To get a list of apps that are available to you.\n    print(apis.api_docs.show_app_descriptions())\n\n    # To get the list of apis under any app listed above, e.g. spotify\n    print(apis.api_docs.show_api_

In [84]:
context_text = [x.content for x in context]

In [94]:
full_text = ' '.join(context_text)

In [96]:
type(full_text)

str

In [88]:
for x in context:
    print(x.role)

user
assistant
user


In [90]:
!pip install transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

[notice] A new release of pip available: 22.3.1 -> 25.3


[notice] To update, run: python.exe -m pip install --upgrade pip

     --------------------------------------- 12.0/12.0 MB 11.3 MB/s eta 0:00:00
     ------------------------------------- 566.1/566.1 kB 11.8 MB/s eta 0:00:00
     ------------------------------------- 277.7/277.7 kB 16.7 MB/s eta 0:00:00
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl (2.7 MB)
     ------------------------------------- 341.4/341.4 kB 22.1 MB/s eta 0:00:00
  Using cached fsspec-2025.12.0-py3-none-any.whl (201 kB)


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [91]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

C:\Users\Aaron McClendon\Documents\github\maml-agent\env311\Lib\site-packages\urllib3\connection.py:779: SystemTimeWarning: System time is way off (before 2025-01-01). This will probably lead to SSL verification errors
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\Aaron McClendon\Documents\github\maml-agent\env311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aaron McClendon\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is 

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [98]:
!pip install torch

[notice] A new release of pip available: 22.3.1 -> 25.3


[notice] To update, run: python.exe -m pip install --upgrade pip


     -------------------------------------- 111.0/111.0 MB 9.0 MB/s eta 0:00:00


  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


In [100]:
import torch
full_ids = tokenizer.encode(' '.join(context_text), return_tensors="pt")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\Aaron McClendon\Documents\github\maml-agent\env311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.